# hw3 - ensembles

## 1 Подготовка данных

Загрузите и предобработайте данные (по своему усмотрению) из hw1

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

path = '../hw1/train_features_with_answers.csv'
df = pd.read_csv(path)
df = df.replace('', np.nan)
y = df['G3']
X = df.drop(columns=['G3'])
num_sel = selector(dtype_include=np.number)
cat_sel = selector(dtype_exclude=np.number)
prep = ColumnTransformer([
    ('num', Pipeline([('im', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_sel),
    ('cat', Pipeline([('im', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_sel)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=17)

def build(model):
    return Pipeline([
        ('prep', prep),
        ('est', model)
    ])


## 2 Обоснуйте выбор слабых (базовых) алгоритмов

Ответ:

Выберу алгоритмы разной природы (линейные, деревья/ансамбли, соседние, ядровые). Они делают разные ошибки — это помогает ансамблю учесть больше шаблонов. Чтобы снизить корреляцию между ними, буду менять случайные подвыборки и гиперпараметры.

## 3 Постройте решение на основе подхода Blending

Правила:
- Нужно использовать вероятности
- Предложите что-то лучше, чем брать среднее от предсказаний моделей (оценивать уверенность алгоритмов, точности и т.д.)
- Заставьте базовые алгоритмы быть некорелированными
- Добавьте рандома (например, стройте ваши алгоритмы на разных выборках, по разному предобрабатывайте данные или применяйте для разных признаков соответствующие алгоритмы ... )
- Проявите смекалку
- Цель: метрика MSE на тесте меньше 10

In [2]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

bases = [
    ('ridge', Ridge(alpha=2.0, random_state=0)),
    ('lasso', Lasso(alpha=0.0005, random_state=1, max_iter=20000)),
    ('rf', RandomForestRegressor(n_estimators=200, random_state=2, n_jobs=-1)),
    ('gbr', GradientBoostingRegressor(learning_rate=0.05, n_estimators=300, random_state=3)),
    ('etr', ExtraTreesRegressor(n_estimators=300, random_state=4, n_jobs=-1)),
    ('knn', KNeighborsRegressor(n_neighbors=5)),
    ('svr', SVR(C=3.0, epsilon=0.1))
]

kf = KFold(n_splits=5, shuffle=True, random_state=123)
oof_errors = {}
fit_models = []
X_tr, X_hold, y_tr, y_hold = train_test_split(X_train, y_train, test_size=0.3, random_state=23)
for name, base in bases:
    oof = np.zeros(len(X_tr))
    for tr_idx, val_idx in kf.split(X_tr):
        Xt, Xv = X_tr.iloc[tr_idx], X_tr.iloc[val_idx]
        yt, yv = y_tr.iloc[tr_idx], y_tr.iloc[val_idx]
        p = build(base)
        p.fit(Xt, yt)
        oof[val_idx] = p.predict(Xv)
    oof_errors[name] = mean_squared_error(y_tr, oof)
    model_full = build(base)
    model_full.fit(X_train, y_train)
    fit_models.append((name, model_full))
errs = np.array([oof_errors[k] for k,_ in fit_models])
w_raw = np.exp(-(errs/np.mean(errs)))
w = w_raw/ w_raw.sum()
bl_pred = np.zeros(len(X_test))
for i,(name, m) in enumerate(fit_models):
    bl_pred += w[i]*m.predict(X_test)
print('MSE blend:', mean_squared_error(y_test, bl_pred))


MSE blend: 7.67188584130622


## 4 Постройте решение на основе подхода Stacking

Правила:
- Реализуйте пайплайн обучения и предсказания (например, sklearn.pipeline или класс)
- Проведите оптимизацию пайплайна
- Оцените вклад каждого базового алгоритма в итоговое предсказание
- Цель: метрика MSE на тесте меньше 10

In [4]:
from sklearn.model_selection import KFold
from sklearn.linear_model import ElasticNetCV

base_list = [
    ('ridge', Ridge(alpha=2.0, random_state=0)),
    ('rf', RandomForestRegressor(n_estimators=250, random_state=5, n_jobs=-1)),
    ('gbr', GradientBoostingRegressor(random_state=6)),
    ('svr', SVR(C=3.0, epsilon=0.1))
]

kf = KFold(n_splits=5, shuffle=True, random_state=321)
Z = np.zeros((len(X_train), len(base_list)))
for j,(nm, est) in enumerate(base_list):
    z = np.zeros(len(X_train))
    for tr, va in kf.split(X_train):
        Xt, Xv = X_train.iloc[tr], X_train.iloc[va]
        yt, yv = y_train.iloc[tr], y_train.iloc[va]
        p = build(est)
        p.fit(Xt, yt)
        z[va] = p.predict(Xv)
    Z[:, j] = z
meta = ElasticNetCV(l1_ratio=[0.1,0.5,0.9], alphas=100, cv=5, random_state=7)
meta.fit(Z, y_train)
P_test = np.column_stack([build(e).fit(X_train, y_train).predict(X_test) for _,e in base_list])
yp = meta.predict(P_test)
print('MSE stack:', mean_squared_error(y_test, yp))


MSE stack: 7.33073072811821


## * Доп задание (не обязательно, но решение будет поощряться)

Правила:
- Постройте несколько сильных алгоритмов разного класса (это может быть бустинг, нейросеть, ансамбль слабых алгоритмов, алгоритм на статистике, что придумаете)
- Реализуйте "управляющий" алгоритм, который на основе входных данных будет выбирать, какой из  сильных алгоритмов запустить (не на основе их работы, а именно на основе данных)